In [ ]:
!pip install transformers datasets scikit-learn torch pandas numpy

In [ ]:
!git clone https://github.com/suu990901/chatgpt-comparison-detection-HC3-Plus.git
%cd chatgpt-comparison-detection-HC3-Plus

In [ ]:
import pandas as pd
import json

file_path = "data/en/train.jsonl"

data = []

with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        data.append(json.loads(line))

df = pd.DataFrame(data)

print("Dataset Loaded Successfully!")
print("Shape:", df.shape)

In [ ]:
# Check original label count
print("Original label count:")
print(df["label"].value_counts())

# Separate labels
human_df = df[df["label"] == 0]
ai_df = df[df["label"] == 1]

# Check if enough samples are available
if len(human_df) >= 25000 and len(ai_df) >= 25000:
    human_sample = human_df.sample(n=25000, random_state=42)
    ai_sample = ai_df.sample(n=25000, random_state=42)

    balanced_df = pd.concat([human_sample, ai_sample])
    balanced_df = balanced_df.sample(frac=1, random_state=42).reset_index(drop=True)

    print("\nBalanced dataset created successfully!")
    print(balanced_df["label"].value_counts())
    print("Final shape:", balanced_df.shape)

else:
    print("\nNot enough samples for 25,000 each.")
    print("Human available:", len(human_df))
    print("AI available:", len(ai_df))

In [ ]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    balanced_df,
    test_size=0.2,
    random_state=42,
    stratify=balanced_df["label"]
)

train_df, val_df = train_test_split(
    train_df,
    test_size=0.1,
    random_state=42,
    stratify=train_df["label"]
)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

In [ ]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

In [ ]:
def tokenize(texts, max_len=128):
    return tokenizer(
        list(texts),
        padding=True,
        truncation=True,
        max_length=max_len,
        return_tensors="pt"
    )

train_encodings = tokenize(train_df["text"])
val_encodings = tokenize(val_df["text"])
test_encodings = tokenize(test_df["text"])

print("Tokenization complete!")

In [ ]:
import torch

class TextDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = list(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

train_dataset = TextDataset(train_encodings, train_df["label"])
val_dataset = TextDataset(val_encodings, val_df["label"])
test_dataset = TextDataset(test_encodings, test_df["label"])

print("Dataset created!")

In [ ]:
from transformers import BertForSequenceClassification

model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

print("BERT model loaded!")

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./bert_results",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./bert_logs",
    load_best_model_at_end=True,
    fp16=True
)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

print("Trainer ready!")

In [ ]:
trainer.train()

In [ ]:
from sklearn.metrics import accuracy_score, classification_report
import numpy as np

predictions = trainer.predict(test_dataset)

preds = np.argmax(predictions.predictions, axis=1)
labels = predictions.label_ids

print("Accuracy:", accuracy_score(labels, preds))
print(classification_report(labels, preds, target_names=["Human", "AI Generated"]))

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

model.save_pretrained("/content/drive/MyDrive/bert_hc3plus_mgd_model")
tokenizer.save_pretrained("/content/drive/MyDrive/bert_hc3plus_mgd_model")

print("BERT model saved!")

In [ ]:
import torch
from transformers import BertTokenizer, BertForSequenceClassification

model_path = "/content/drive/MyDrive/bert_hc3plus_mgd_model"

tokenizer = BertTokenizer.from_pretrained(model_path)
model = BertForSequenceClassification.from_pretrained(model_path)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

def predict_text(text):
    if len(text.split()) < 10:
        print("⚠️ Text too short! Please enter at least 10 words.")
        return

    inputs = tokenizer(
        text,
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        prediction = torch.argmax(outputs.logits, dim=1).item()
        confidence = torch.softmax(outputs.logits, dim=1).max().item()

    label = "🤖 AI Generated" if prediction == 1 else "👤 Human Written"

    print("Prediction:", label)
    print(f"Confidence: {confidence * 100:.2f}%")

while True:
    text = input("\nEnter text to check or type quit: ")

    if text.lower() == "quit":
        print("Exiting...")
        break

    predict_text(text)